<a href="https://colab.research.google.com/github/en970/gausscapture/blob/main/notebooks/GaussCapture_Colab_Trainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GaussCapture · Colab trainer

Trains a 3D Gaussian splat from a `dataset.zip` produced by
`gausscapture colab <project>`.

**Runtime → Change runtime type → GPU.** A T4 is enough. ~30 minutes.

---

### Why gsplat and not the reference implementation

The original 3D Gaussian Splatting code from Inria is licensed for
**non-commercial research only**, and anything trained with it inherits that
restriction — which would make GaussCapture's MIT licence a promise it cannot
keep. [gsplat](https://github.com/nerfstudio-project/gsplat) is Apache-2.0 and
produces equivalent quality. See `docs/DEPENDENCIES.md`.

### Why the setup below looks fussy

Every workaround here was earned by a failure, and each one fails *silently* or
*misleadingly* if skipped. They are commented individually.


## 1 · GPU and Drive

Output goes to Google Drive, not `/content`. Colab recycles idle runtimes and
wipes `/content` when it does — a completed training run was lost that way.

**Put `dataset_A_good.zip` in the root of your Drive first** (drag it onto
drive.google.com). Colab's own upload widget is avoided: it breaks when the
cell output is restored from an earlier browser session.


In [ ]:
import torch
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU'
print(f'{torch.cuda.get_device_name(0)} · torch {torch.__version__} · cuda {torch.version.cuda}')

from google.colab import drive
drive.mount('/content/drive')

import pathlib, shutil, zipfile
DRIVE = pathlib.Path('/content/drive/MyDrive')
OUT = DRIVE / 'gausscapture_result'   # survives a recycled runtime
OUT.mkdir(exist_ok=True)

found = sorted(DRIVE.glob('dataset*.zip'))
assert found, 'Put dataset_A_good.zip in the root of your Drive, then rerun.'

DATA = pathlib.Path('/content/dataset')
if DATA.exists():
    shutil.rmtree(DATA)
DATA.mkdir()
with zipfile.ZipFile(found[0]) as z:
    z.extractall(DATA)
print(f"{len(list((DATA / 'images').glob('*.jpg')))} images · model "
      f"{sorted(p.name for p in (DATA / 'sparse' / '0').iterdir())}")


## 2 · Install

Packages are installed **one at a time**. `pip install -r requirements.txt` is
all-or-nothing: one package that fails to build takes the whole list with it,
and the first symptom is an unrelated `ModuleNotFoundError` much later.

Two of them must come from git rather than PyPI:

* **`pycolmap`** — PyPI's package of that name is the official COLMAP binding
  and has no `SceneManager`. gsplat wants a different, older project that
  happens to share the name.
* **`nerfview`** — the pinned commit; the PyPI release misses a dependency.

`fused-ssim` builds CUDA and is usually what breaks a bulk install.


In [ ]:
import subprocess, sys

# HuggingFace 'datasets' shadows gsplat's examples/datasets/. That directory has
# no __init__.py, making it a namespace package -- and a regular package in
# site-packages beats a namespace portion no matter what the path order is.
# The wrong pycolmap has to go for the same reason: same name, different project.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y',
                'datasets', 'pycolmap'])

PACKAGES = [
    'gsplat',            # PyPI build compiles its kernels on first use, not at install
    'viser', 'splines', 'imageio[ffmpeg]', 'scikit-learn', 'tqdm',
    'torchmetrics[image]', 'opencv-python', 'tyro>=0.8.8', 'Pillow',
    'tensorboard', 'tensorly', 'pyyaml', 'matplotlib',
    'git+https://github.com/rmbrualla/pycolmap@cc7ea4b7301720ac29287dbe450952511b32125e',
    'git+https://github.com/nerfstudio-project/nerfview@4538024fe0d15fd1a0e4d760f3695fc44ca72787',
    'git+https://github.com/rahul-goel/fused-ssim@328dc9836f513d00c4b5bc38fe30478b4435cbb5',
]
for package in PACKAGES:
    done = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package],
                          capture_output=True, text=True)
    label = package.split('@')[0].split('/')[-1] if package.startswith('git+') else package
    print(f"  {'OK  ' if done.returncode == 0 else 'FAIL'} {label}")
    if done.returncode != 0:
        print('       ', done.stderr.strip()[-400:])

# numpy 1.x wrapped np.uint64(-1) round to 2**64-1; numpy 2 raises instead. This
# old pycolmap relies on the wrap. Same value, spelled in a way numpy 2 accepts --
# far less disruptive than downgrading numpy under everything else in Colab.
import pathlib
for scene_manager in pathlib.Path('/usr/local/lib').rglob('pycolmap/scene_manager.py'):
    text = scene_manager.read_text()
    if 'np.uint64(-1)' in text:
        scene_manager.write_text(text.replace('np.uint64(-1)', 'np.uint64(2**64 - 1)'))
        print('patched', scene_manager)


## 3 · Example scripts, matched to the installed library

`simple_trainer.py` lives in the repository, not the package, and `main`'s copy
imports modules the released library does not have. Cloning the tag that matches
the installed version is what keeps the two in step.


In [ ]:
import pathlib, shutil, subprocess
import gsplat

SRC = pathlib.Path('/content/gsplat_pinned')
if SRC.exists():
    shutil.rmtree(SRC)
subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch',
                f'v{gsplat.__version__}',
                'https://github.com/nerfstudio-project/gsplat.git', str(SRC)],
               check=True)
EXAMPLES = SRC / 'examples'
print(f'gsplat {gsplat.__version__} · examples from tag v{gsplat.__version__}')

# Import everything the trainer needs before spending half an hour on it.
check = subprocess.run(
    [sys.executable, '-c',
     'import numpy, gsplat, viser, nerfview, torchmetrics, fused_ssim\n'
     'from pycolmap import SceneManager\n'
     'print("imports OK · numpy", numpy.__version__)'],
    capture_output=True, text=True, cwd=str(EXAMPLES))
print(check.stdout)
if check.returncode != 0:
    print(check.stderr)          # never truncated: the tail is where the cause is
    raise SystemExit('fix the import above before training')


## 4 · Train

`mcmc` densification suits the uneven coverage of a walked capture. Output
streams live, and lands in Drive, so neither a long silence nor a recycled
runtime costs you the run.


In [ ]:
STEPS = 30_000      # 7_000 for a quick look; reaches most of the quality
DOWNSCALE = 1       # raise to 2 if the GPU runs out of memory

command = [sys.executable, '-u', str(EXAMPLES / 'simple_trainer.py'), 'mcmc',
           '--data_dir', str(DATA),
           '--data_factor', str(DOWNSCALE),
           '--result_dir', str(OUT),
           '--max_steps', str(STEPS),
           '--save_ply', '--disable_viewer']
print('$', ' '.join(command), end='\n\n')

# -u plus line-buffered Popen: progress appears as it happens rather than in one
# block at the end, so a half-hour run does not look like a hang.
process = subprocess.Popen(command, cwd=str(EXAMPLES), stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
print('\nDONE' if process.wait() == 0 else '\nFAILED')


## 5 · Collect

The `.ply` is already in Drive and stays there. Downloading is only for
convenience — view it in [SuperSplat](https://superspl.at/editor), or import it
with `gausscapture report --splat <file.ply>` to see it beside the sparse cloud
it was trained from.


In [ ]:
plys = sorted(OUT.rglob('*.ply'), key=lambda p: p.stat().st_size)
for p in plys:
    print(f'  {p.relative_to(OUT)}  {p.stat().st_size / 1e6:.1f} MB')
assert plys, 'No .ply — read the training log above.'

splat = plys[-1]
print(f'\nlargest: {splat}')
from google.colab import files
files.download(str(splat))
